In [1]:
#File opening
import tarfile
import subprocess

#Data processing
import json
import tkinter as tk
from tkinter import ttk

In [2]:
def iter_in_file(tar_name, file_name):
    if not tarfile.is_tarfile(tar_name):
        print(tar_name, "isn't a tar xz file!")
        exit()
    xz_call = subprocess.Popen(
        ["xz", "-dc", "-T10", tar_name], #The command here is xz --decompress --stdout "tar file"
        stdout = subprocess.PIPE
    )

    tar_call = subprocess.Popen(
        ["tar", "-xO", "--file=-", file_name], #The command here is tar -x0 --file="the file inside thar"
        stdin = xz_call.stdout,
        stdout = subprocess.PIPE
    )

    xz_call.stdout.close()

#    while True:
#        line = tar_call.stdout.readline()
    for index, line_bytes in enumerate(tar_call.stdout):
        line = line_bytes.decode("utf-8", errors = "replace").rstrip("\n")
        dictionary = json.loads(line_bytes)
        yield index, dictionary

    print("Closing subprocesses and pipes")
    tar_call.stdout.close()
    tar_call.wait()
    xz_call.wait()

In [3]:
def populate_tree(tree, data, parent=""):
    if isinstance(data, dict):
        for key, value in data.items():
            node = tree.insert(parent, "end", text=str(key))
            populate_tree(tree, value, node)
    elif isinstance(data, list):
        for index, value in enumerate(data):
            node = tree.insert(parent, "end", text=f"[{index}]")
            populate_tree(tree, value, node)
    else:
        tree.insert(parent, "end", text=str(data))

def display_json(data):
    root = tk.Tk()
    root.title("JSON Viewer")
    root.geometry("600x400")

    frame = tk.Frame(root)
    frame.pack(fill="both", expand=True, padx=10, pady=10)

    tree = ttk.Treeview(frame, columns=("Values",), show="tree")
    tree.heading("#0", text="Key / Index")
    tree.pack(fill="both", expand=True, side="left")

    populate_tree(tree, data)

    root.mainloop()

In [ ]:
# ARTISTS

for index, dictionary in iter_in_file("dumps/artist.tar.xz", "mbdump/artist"):
    show = False
    for i, relation in enumerate(dictionary.get("relations", [])):
        if relation.get("artist") and relation.get("type") == "collaboration" and (relation.get("start") or relation.get("end")):
            print(i)
            print(relation.get("attributes"))
            show = True
    if show:
        display_json(dictionary)
        print("---------------")

0
[]
---------------
0
[]
---------------
0
[]
---------------
0
[]


In [ ]:
INDEX = 1
for index, dictionary in iter_in_file("dumps/release.tar.xz", "mbdump/release"):
    show = False
    for relation in dictionary["relations"]:
        if "release" in relation and not (relation["type"] in ["transl-tracklisting", "remaster", "replaced by", "supporting release", "part of set"]):
            show = True
    if show:
        display_json(dictionary)

KeyboardInterrupt: 

In [ ]:
#IMAGINE

INDEX = 1
for index, dictionary in iter_in_file("dumps/work.tar.xz", "mbdump/work"):
    if dictionary["id"] != "517ff96c-5970-3eac-9ae5-bcafc15f5bc7":
        continue
    for i, relation in enumerate(dictionary["relations"]):
        if "artist" in relation:
            print(i, relation["type"])
    display_json(dictionary)
    with open("imagine.json", "w") as f:
        json.dump(dictionary, f)

    

0 composer
1 lyricist
2 lyricist
Closing subprocesses and pipes


In [ ]:
# recording release relationship

INDEX = 1
for index, dictionary in iter_in_file("dumps/release.tar.xz", "mbdump/release"):
    show = False
    for media in dictionary.get("media", []):
        for i, track in enumerate(media.get("tracks", [])):
            if track.get("recording", None):
                recording = track["recording"]
                for j, relation in enumerate(recording["relations"]):
                    if "release" in relation:
                        show = True
                        print(i, j)
    if show:
        display_json(dictionary)

4 25
5 14
12 7
14 3


KeyboardInterrupt: 

In [ ]:
#RELEASE GROUP
INDEX = 1
for index, dictionary in iter_in_file("dumps/release-group.tar.xz", "mbdump/release-group"):
    if dictionary["id"] != "5bc75318-5928-4ad0-8be0-83a89d13fed2":
        continue
    # show = False
    show = True
    # for media in dictionary.get("media", []):
    #     for i, track in enumerate(media.get("tracks", [])):
    #         if track.get("recording", None):
    #             recording = track["recording"]
    #             for j, relation in enumerate(recording["relations"]):
    #                 if "release" in relation:
    #                     show = True
    #                     print(i, j)
    if show:
        display_json(dictionary)
#    if index > 1:  
#        break
#    with open("back_in_the_saddle_bach.json", "w") as f:
#        json.dump(dictionary, f)


KeyboardInterrupt: 

In [ ]:
#LEMONADE
INDEX = 1
for index, dictionary in iter_in_file("dumps/release.tar.xz", "mbdump/release"):
    if dictionary["id"] != "a32958a5-848a-4175-a0f0-a74f032c2ead":
        continue
    display_json(dictionary)
    with open("lemonade.json", "w") as f:
        json.dump(dictionary, f)

In [ ]:
# SEARCH FOR DATE
INDEX = 1
for index, dictionary in iter_in_file("dumps/release.tar.xz", "mbdump/release"):
    for media in dictionary.get("media", []):
        for track in media.get("tracks", []):
            recording = track.get("recording")
            if recording and recording.get("date"):
                    print("has date!")